In [21]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text

In [22]:
import pandas as pd

# Load the dataset

df = pd.read_csv("datasets/Customer_Support_Tickets_Cleaned.csv")

In [23]:
df = df[['clean_sub_body', 'department']]

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df['clean_sub_body']
le = LabelEncoder()

y = le.fit_transform(df['department'])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2, stratify=y)

In [25]:
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4")

In [26]:
def get_sentence_embeding(sentences):
    preprocessed_text = bert_preprocess(sentences)
    return bert_encoder(preprocessed_text)['pooled_output']

get_sentence_embeding([
    "500$ discount. hurry up", 
    "Bhavin, are you up for a volleybal game tomorrow?"]
)

<tf.Tensor: shape=(2, 768), dtype=float32, numpy=
array([[-0.8436513 , -0.5135406 , -0.88872325, ..., -0.7478876 ,
        -0.7532389 ,  0.91978955],
       [-0.87209153, -0.5054329 , -0.94447136, ..., -0.8585005 ,
        -0.71741897,  0.8808778 ]], dtype=float32)>

In [27]:
# BERT layers
text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessed_text = bert_preprocess(text_input)
bert_encoder.trainable = False
outputs = bert_encoder(preprocessed_text)

# Classification head
l = tf.keras.layers.Dropout(
    0.1,
    name="dropout"
)(outputs['pooled_output'])

l = tf.keras.layers.Dense(
    len(le.classes_),
    activation='softmax',
    name="output"
)(l)

# Model
model = tf.keras.Model(
    inputs=text_input,
    outputs=l
)

METRICS = [
    tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')
]

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=METRICS
)

model.fit(X_train, y_train, epochs=10)

Epoch 1/10
1545/1545 [==============================] - 372s 236ms/step - loss: 1.1777 - accuracy: 0.5978
Epoch 2/10
1545/1545 [==============================] - 381s 246ms/step - loss: 1.1094 - accuracy: 0.6228
Epoch 3/10
1545/1545 [==============================] - 364s 236ms/step - loss: 1.0907 - accuracy: 0.6281
Epoch 4/10
1545/1545 [==============================] - 369s 239ms/step - loss: 1.0826 - accuracy: 0.6320
Epoch 5/10
1545/1545 [==============================] - 362s 234ms/step - loss: 1.0775 - accuracy: 0.6340
Epoch 6/10
1545/1545 [==============================] - 363s 235ms/step - loss: 1.0698 - accuracy: 0.6388
Epoch 7/10
1545/1545 [==============================] - 361s 234ms/step - loss: 1.0674 - accuracy: 0.6381
Epoch 8/10
1545/1545 [==============================] - 363s 235ms/step - loss: 1.0647 - accuracy: 0.6385
Epoch 9/10
1545/1545 [==============================] - 365s 236ms/step - loss: 1.0630 - accuracy: 0.6400
Epoch 10/10
1545/1545 [=======================